# Sinh dữ liệu Gaussian 2 chiều (TV1)

Hàm mục tiêu:

$$f(\mathbf{x}) = \frac{1}{2\pi\sqrt{\det C}} \exp\left(-\tfrac{1}{2}(\mathbf{x}-\boldsymbol\mu)^T C^{-1}(\mathbf{x}-\boldsymbol\mu)\right),\quad
\boldsymbol\mu = \begin{bmatrix}0\\0\end{bmatrix},\ C = \begin{bmatrix}0.3 & -0.5\\-0.5 & 2\end{bmatrix}$$

Miền lấy mẫu: $-2 \le x_1 \le 2$, $-4 \le x_2 \le 4$, phân bố đều.

| Tập | Số mẫu | Vai trò |
|---|---|---|
| `train_pool` | 2000 | Nguồn train; Exp A lấy tập con lồng nhau `pool[:A]` |
| `val` | 200 | Early stopping, checkpoint, chọn mô hình |
| `test` | 100 | Chỉ dùng để đánh giá cuối, không dùng chọn cấu hình |

Ba tập được rút liên tiếp từ cùng một `numpy.random.default_rng(42)` theo thứ tự train_pool → val → test.
Dữ liệu chỉ sinh **một lần**; nếu file đã tồn tại, notebook chỉ kiểm tra lại rằng sinh lại cho ra đúng dữ liệu cũ.

In [1]:
import hashlib, json, sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import integrate
from scipy.stats import multivariate_normal

ROOT = Path.cwd() if (Path.cwd() / "preprocessing").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from preprocessing.gaussian import (COV, MU, DATA_SEED, SPLIT_SIZES, X1_RANGE, X2_RANGE,
                                    gaussian_2d, generate_all, save_splits)

DATA_DIR = ROOT / "data"
print("ROOT =", ROOT.name, "| DATA_DIR =", DATA_DIR.relative_to(ROOT))

ROOT = BTL | DATA_DIR = data


## 1. Kiểm tra hàm Gaussian

In [2]:
det_c = np.linalg.det(COV)
print(f"det(C) = {det_c:.4f}")
print(f"C^-1 =\n{np.linalg.inv(COV)}")
print(f"C xác định dương: trị riêng = {np.linalg.eigvalsh(COV)}")

f00 = gaussian_2d([0.0, 0.0])[0]
print(f"f(0, 0) = {f00:.6f}  (lý thuyết 1/(2π·sqrt(0.35)) = {1 / (2 * np.pi * np.sqrt(0.35)):.6f})")
assert abs(f00 - 0.269) < 1e-3

# So với scipy trên 10 000 điểm ngẫu nhiên
pts = np.random.default_rng(0).uniform([-2, -4], [2, 4], size=(10_000, 2))
max_err = np.max(np.abs(gaussian_2d(pts) - multivariate_normal(MU, COV).pdf(pts)))
print(f"Sai lệch lớn nhất so với scipy: {max_err:.2e}")
assert max_err < 1e-12

det(C) = 0.3500
C^-1 =
[[5.71428571 1.42857143]
 [1.42857143 0.85714286]]
C xác định dương: trị riêng = [0.16384585 2.13615415]
f(0, 0) = 0.269021  (lý thuyết 1/(2π·sqrt(0.35)) = 0.269021)
Sai lệch lớn nhất so với scipy: 1.11e-16


In [3]:
# Tích phân trên toàn R^2 phải bằng 1; tích phân trên miền đề bài cho biết phần khối lượng nằm trong miền
f_scalar = lambda x2, x1: gaussian_2d([x1, x2])[0]
total, _ = integrate.dblquad(f_scalar, -10, 10, -20, 20)
inside, _ = integrate.dblquad(f_scalar, *X1_RANGE, *X2_RANGE)
print(f"Tích phân trên R^2 ≈ {total:.6f}")
print(f"Tích phân trên miền [-2,2]x[-4,4] ≈ {inside:.6f}")

Tích phân trên R^2 ≈ 1.000000
Tích phân trên miền [-2,2]x[-4,4] ≈ 0.995151


## 2. Sinh và lưu dữ liệu

In [4]:
splits = generate_all()
paths = {name: DATA_DIR / f"{name}.csv" for name in SPLIT_SIZES}

if all(p.exists() for p in paths.values()):
    # Đã có dữ liệu: không ghi đè, chỉ kiểm tra tái lập
    for name, p in paths.items():
        saved = pd.read_csv(p)
        np.testing.assert_allclose(saved.to_numpy(), splits[name].to_numpy(), rtol=0, atol=1e-15)
    print("Dữ liệu đã tồn tại và khớp với seed", DATA_SEED, "-> không ghi đè.")
else:
    save_splits(splits, DATA_DIR)
    print("Đã lưu:", [p.relative_to(ROOT).as_posix() for p in paths.values()])

Dữ liệu đã tồn tại và khớp với seed 42 -> không ghi đè.


## 3. Kiểm tra dữ liệu đã lưu

In [5]:
data = {name: pd.read_csv(p) for name, p in paths.items()}
rows = []
for name, df in data.items():
    rows.append({
        "tập": name, "n": len(df), "NaN": int(df.isna().sum().sum()),
        "x1_min": df.x1.min(), "x1_max": df.x1.max(),
        "x2_min": df.x2.min(), "x2_max": df.x2.max(),
        "y_min": df.y.min(), "y_max": df.y.max(),
        "y khớp f(x)": np.allclose(df.y, gaussian_2d(df[["x1", "x2"]].to_numpy()), rtol=1e-12, atol=0),
    })
    assert df.x1.between(*X1_RANGE).all() and df.x2.between(*X2_RANGE).all()
pd.DataFrame(rows).set_index("tập")

,n,NaN,x1_min,x1_max,x2_min,x2_max,y_min,y_max,y khớp f(x)
tập,,,,,,,,,
train_pool,2000,0,-1.997925,1.997688,-3.994862,3.998123,1.236383e-13,0.267541,True
val,200,0,-1.974701,1.986021,-3.975511,3.986048,2.168712e-12,0.266054,True
test,100,0,-1.991059,1.984509,-3.986953,3.921165,2.133066e-12,0.249406,True


In [6]:
# Kiểm tra rò rỉ: không có điểm x nào trùng giữa các tập (kể cả trùng trong cùng tập)
keys = {name: set(map(tuple, df[["x1", "x2"]].to_numpy())) for name, df in data.items()}
for name, df in data.items():
    assert len(keys[name]) == len(df), f"{name} có điểm trùng"
names = list(keys)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        common = keys[names[i]] & keys[names[j]]
        print(f"{names[i]} ∩ {names[j]}: {len(common)} điểm chung")
        assert not common

train_pool ∩ val: 0 điểm chung
train_pool ∩ test: 0 điểm chung
val ∩ test: 0 điểm chung


In [7]:
# Metadata để truy vết: seed, kích thước, mã băm MD5 của từng file
meta = {
    "seed": DATA_SEED,
    "rng": "numpy.random.default_rng",
    "generation_order": list(SPLIT_SIZES),
    "sizes": SPLIT_SIZES,
    "mu": MU.tolist(), "cov": COV.tolist(),
    "x1_range": list(X1_RANGE), "x2_range": list(X2_RANGE),
    "sampling": "uniform",
    "md5": {p.name: hashlib.md5(p.read_bytes().replace(b"\r\n", b"\n")).hexdigest() for p in paths.values()},
}
meta_path = DATA_DIR / "metadata.json"
if meta_path.exists():
    assert json.loads(meta_path.read_text(encoding="utf-8"))["md5"] == meta["md5"], "CSV đã bị thay đổi!"
else:
    meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(meta["md5"], indent=2))

{
  "train_pool.csv": "d5002417dfd0b7b9020167e67289c261",
  "val.csv": "386275d59879c9907ff47402e70fd350",
  "test.csv": "36871f529de6b7182f2c23674de368ee"
}
